# Abliteration in the Cloud

This notebook will guide you through setting up the environment, deploying the `abliteration` repository from GitHub, downloading a target model, applying the abliteration process to remove refusal behavior, generating documentation, and then pushing the resulting modified model back up to the Hugging Face hub.

## 1. Environment Setup & Deployment

First, we'll clone the repository and install all of its dependencies using `uv`.

In [ ]:
!git clone https://github.com/cs2764/abliteration.git
%cd abliteration

# Install uv (fast python package installer)
!pip install uv

# Install project requirements utilizing uv to speed up the process
!uv pip install --system -r requirements.txt

# Install huggingface_hub for model downloading/uploading
!uv pip install --system -U huggingface_hub

## 2. Secure Hugging Face Authentication

In order to both download gated models (if applicable) and explicitly upload our processed models to our Hugging Face account, we need to log in using an access token.

You can get an access token from [Hugging Face Tokens](https://huggingface.co/settings/tokens).
- Ensure your token is a **WRITE** token.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 3. Configure Download Path

We need to ensure all Hugging Face operations are kept strictly local rather than in the shared system cache, giving you direct access to the files.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HOME"] = "./hf_cache" # Force cache to local folder

# Set base model and targeted local output
MODEL_ID = "Qwen/Qwen3.5-35B-A3B"
LOCAL_DIR = f"./models/{MODEL_ID.split('/')[-1]}"

!hf download {MODEL_ID} --local-dir {LOCAL_DIR}
print(f"\nModel downloaded strictly to: {LOCAL_DIR}")

## 4. Prepare Configuration

Prepare our configuration YAML. Here we generate `config.yaml` using Python inline, ensuring it points to our newly downloaded model.

In [ ]:
yaml_config = f"""
model: "{LOCAL_DIR}"
output_dir: "{LOCAL_DIR}-abliterated"

inference:
  device: "cuda"
  batch_size: 2
  max_length: 512
  flash_attn: true # Faster processing on compatible GPUs (A100, etc.)
  load_in_8bit: true # Utilize 8-bit quantization to fit larger models into available VRAM

measurements:
  harmful_prompts: "./data/harmful.parquet"
  harmless_prompts: "./data/harmless.parquet"
  clip: 1.0

ablation:
  method: "full"
  sparsify_method: "percentile"
  quantile: 0.995
  magnitude_threshold: 0.05
  top_k: 48
  global_scale: 1.5
  layer_overrides: {{}}
"""

with open('config_colab.yaml', 'w') as f:
    f.write(yaml_config)

print("Config saved as `config_colab.yaml`")

## 5. Execute Abliteration

Run the core abliteration process. Depending on model size and GPU, this may take a while.

In [ ]:
!python abliterate.py config_colab.yaml

## 6. Test Model Locally (Optional)

Before uploading, you can use the built-in `chat.py` utility to run a test on the newly minted model to verify behavior.

In [ ]:
# Warning: Running interactive prompts inside a notebook cell might be tricky 
# or cause timeouts. Using a predetermined prompt or script snippet is safer.

!python chat.py -m {LOCAL_DIR}-abliterated

## 7. Generate Model README

Before uploading, we automatically generate a clean `README.md` file for your model in English, giving attribution to the base model and the methodology's implementation repository.

In [ ]:
readme_content = f"""# {MODEL_ID.split('/')[-1]}-abliterated\n
\n
This is an abliterated version of the original [{MODEL_ID}](https://huggingface.co/{MODEL_ID}) model.\n
It was created to remove refusal behaviors based on the methodology from [Orion-zhen/abliteration](https://github.com/Orion-zhen/abliteration), utilizing the modified custom implementation available at [cs2764/abliteration](https://github.com/cs2764/abliteration).\n
"""

readme_path = f"{LOCAL_DIR}-abliterated/README.md"
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

print(f"Model README generated successfully at {readme_path}!")

## 8. Upload the Modified Model to Hugging Face

Finally, push your new model upwards. Be sure to supply **your desired repository ID**.

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()

# ---------------- ACCOUNT CONFIGURATION ----------------
HF_USERNAME = "cs2764"
repo_id = f"{HF_USERNAME}/{MODEL_ID.split('/')[-1]}-abliterated"
# -------------------------------------------------------

# Creates the repo if it does not exist
api.create_repo(repo_id=repo_id, exist_ok=True)

print(f"Uploading to {repo_id}...")
api.upload_folder(
    folder_path=f"{LOCAL_DIR}-abliterated",
    repo_id=repo_id,
    repo_type="model",
)

print("Upload complete!")